# Father — Disk Investigation

Read-only Sleuth Kit examination of one Father (`userland_father_ldpreload`)
scenario run's acquired disk image. This notebook is the **single canonical
disk workflow** for the Father scenario: it supersedes the earlier
`runme_disk.sh` shell script and `disk_notebook.md` documentation-only
notebook (both deprecated alongside this notebook — see
`investigations/father/README.md`).

**What this notebook does:** invoke TSK/ext4 forensic tools via `subprocess`
(never shell strings), capture raw output under this run's own derived
output directory, parse the small pieces of that output needed to build one
canonical `findings` dictionary, validate the preserved manual recovery
artifacts required by this host, compute metrics from `findings` (never from
re-running a tool), and render the final report from `findings`.

**What this notebook does not do:** reimplement any forensic tool in
Python, or claim a result it has not actually produced this run.

Sections, in execution order:

1. Case configuration and `RUN_ID`
2. Path validation and acquisition integrity
3. Command execution methodology
4 & 5. Partition and root filesystem discovery
6. `/etc/ld.so.preload` investigation
7. Installed library path, identity, and hash
8. `/tmp` artifact investigation
9. Deleted `/tmp/rk.so` metadata check
10. Targeted `extundelete` recovery
11. Unallocated space and PhotoRec carve
12. ext4 journal metadata
13. Bounded supporting inventories (shell history, `/var/log`)
14. Final file context and timestamp review
15. Limitations, metrics, and final report

## 1. Case configuration and RUN_ID

`RUN_ID` is a plain Python constant set in the first code cell (default:
the current accepted Father run), overridable via the `RUN_ID` environment
variable for `jupyter nbconvert --execute`. There is no shell `export`
propagation anywhere in this workflow -- `RUN_ID` is used by every later
cell, and re-running the notebook top-to-bottom (Kernel → Restart & Run
All) is the supported way to re-run the whole investigation.

The first code cell resolves `REPO_ROOT` so the notebook works both from
the repository root (`jupyter nbconvert --execute`) and run interactively
cell-by-cell, where the working directory is normally the notebook's own
directory (`investigations/father/`).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# ## 1. Case configuration and RUN_ID
#
# `RUN_ID` is a plain Python constant set below (default: the current 
# accepted Father run), overridable via the `RUN_ID` environment variable 
# for `jupyter nbconvert --execute`. There is no shell `export` propagation 
# -- `RUN_ID` is used by every later cell, and re-running the notebook 
# top-to-bottom is the supported way to re-run the investigation.
#
# This cell also resolves `REPO_ROOT` so the notebook works both from
# the repository root and interactively cell-by-cell.

# Change this to investigate a different Father run. The RUN_ID env var
# overrides it for `jupyter nbconvert --execute`, which can't edit cells.
RUN_ID = os.environ.get("RUN_ID", "father-u22-20260820-01")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "shared" / "experiments").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent  # interactive: cwd is investigations/father/
assert (
    REPO_ROOT / "shared" / "experiments"
).is_dir(), "run from repo root or investigations/father/"

sys.path.insert(
    0, str(REPO_ROOT / "investigations" / "father")
)  # so investigation_utils imports

from investigation_utils import (
    resolve_run_paths,
    ensure_output_dirs,
    save_raw,
    write_json,
    safe_sha256,
    parse_mmls_root_offset,
    parse_ewfverify,
    parse_fsstat,
    parse_istat,
    parse_fls_regular_files,
    detect_timestomp,
    run_command,
    write_report,
)

# Scenario configuration:
# expected targets and markers defined by the Father scenario.
EXPECTED_TMP_ARTIFACTS = {"__malicious_recon", "__malicious_harvest"}
JOURNAL_SEARCH_MARKERS = ["rk.so", "__malicious_recon", "__malicious_harvest", "selinux.so.3", "ld.so.preload"]
DELETED_FILE_RM_COMMAND = "rm -f -- /tmp/rk.so"

print(f"RUN_ID = {RUN_ID}")

## 2. Path validation and acquisition integrity

Every path is derived from `RUN_ID` alone (`resolve_run_paths`), never
hardcoded. `resolve_run_paths` raises immediately if a required input file
under `shared/experiments/<RUN_ID>/` is missing, before any output
directory is created — this notebook never writes into the acquisition
input tree.

Integrity is checked the standard forensic way: `ewfverify` is re-run
against the image now, independently of the acquisition-time sidecar, and
its computed SHA-256 is compared against the value recorded in
`dumps/acquisition.json`.

In [ ]:
# ## 2. Path validation and acquisition integrity
# 
# Resolves paths from `RUN_ID`, ensures output directories exist, and 
# verifies the integrity of the E01 acquisition via `ewfverify`.

paths = resolve_run_paths(RUN_ID, repo_root=REPO_ROOT)
ensure_output_dirs(paths)

with open(paths.manifest, "r") as f:
    manifest = json.load(f)
with open(paths.acquisition, "r") as f:
    acquisition = json.load(f)

# Acquisition integrity: 
# ewfverify is the standard tool for E01 files.
verify_proc = run_command(
    ["ewfverify", "-d", "sha256", "-x", str(paths.disk_e01)],
    label="01-ewfverify.txt", paths=paths
)

verify_info = parse_ewfverify(verify_proc.stdout)
print(f"SHA256 (from ewfverify): {verify_info['computed_hash']}")
assert verify_info["success"], "Acquisition integrity check failed"
print("SHA256 matches metadata:", verify_info["computed_hash"] == acquisition["disk"]["sha256"])

## 3. Command execution methodology

This notebook follows a strict "Visible Command" methodology.

1. Every tool is invoked using `run_command(args, label, paths)`, which 
   wraps `subprocess.run(list_of_args)`.
2. `stdout` and `stderr` are captured and returned.
3. The function automatically writes the tool's exact output to 
   `derived/disk/raw/<label>` for audit.
4. It also appends the command line and exit code to 
   `logs/disk-commands.log`.
5. Small, specific parsers in `investigation_utils.py` turn raw text into 
   Python variables for interpretation.

## 4 & 5. Partition discovery and Root validation

In [ ]:
# 4. mmls: identify partition layout.
mmls_proc = run_command(
    ["mmls", str(paths.disk_e01)],
    label="02-mmls.txt", paths=paths
)

offset_sector = parse_mmls_root_offset(mmls_proc.stdout)
print("Root partition offset (sectors):", offset_sector)

# 5. fsstat: validate filesystem and block size.
fsstat_proc = run_command(
    ["fsstat", "-o", offset_sector, str(paths.disk_e01)],
    label="03-fsstat.txt", paths=paths
)

fs_info = parse_fsstat(fsstat_proc.stdout)
assert fs_info["fs_type"] == "Ext4", f"Expected Ext4, found {fs_info['fs_type']}"
print("Confirmed filesystem info:", fs_info)

## 6. /etc/ld.so.preload investigation

In [ ]:
# Resolve the path to an inode on this run's own image.'
ifind_proc = run_command(
    ["ifind", "-o", offset_sector, "-n", "/etc/ld.so.preload", str(paths.disk_e01)],
    label="04-ifind-preload.txt", paths=paths
)
preload_inode = ifind_proc.stdout.strip()
assert preload_inode, "/etc/ld.so.preload did not resolve to an inode on this image"
print("/etc/ld.so.preload inode:", preload_inode)

# icat reads its content (the installed library's path).
icat_proc = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), preload_inode],
    label="05-ld.so.preload-content.txt", paths=paths
)
preload_content = icat_proc.stdout
print("content:", repr(preload_content))

# istat reads its MAC times.
istat_proc = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), preload_inode],
    label="06-istat-preload.txt", paths=paths
)
preload_istat = parse_istat(istat_proc.stdout)
print("istat:", preload_istat)

## 7. Installed library path, identity, and hash

Resolve the library named by `/etc/ld.so.preload` to its inode on this run's
own image, extract its bytes with `icat`, and confirm identity by SHA-256
against the manifest's known-good build input. Static characterisation
(`file`/`strings`) and the MAC-timestamp / timestomp review are deferred to
Section 14, after the persistence, recovery, and log checks.

In [ ]:
# Discover /lib symlink target from this run's own evidence.
lib_dir_ifind = run_command(
    ["ifind", "-o", offset_sector, "-n", "/lib", str(paths.disk_e01)],
    label="07a-ifind-lib-dir.txt", paths=paths
)
lib_dir_inode = lib_dir_ifind.stdout.strip()
assert lib_dir_inode, "/lib did not resolve to an inode on this image"

lib_dir_istat = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), lib_dir_inode],
    label="07b-istat-lib-dir.txt", paths=paths
)
lib_dir_info = parse_istat(lib_dir_istat.stdout)
print("/lib symlink target:", lib_dir_info["symlink_target"])

# Resolve library path (handling the /lib symlink discovered above).
lib_path = preload_content.strip()
if lib_dir_info["symlink_target"]:
    # Handle Usr-Merge (e.g. /lib -> /usr/lib) by resolving symlink target.
    lib_name = lib_path.split("/")[-1]
    lib_path_resolved = "/" + lib_dir_info["symlink_target"] + "/" + lib_name
else:
    lib_path_resolved = lib_path
print(f"Resolved library path: {lib_path_resolved}")

# Path Resolution -- locate the installed library inode by resolved path.
lib_ifind = run_command(
    ["ifind", "-o", offset_sector, "-n", lib_path_resolved, str(paths.disk_e01)],
    label="08-ifind-lib.txt", paths=paths
)
lib_inode = lib_ifind.stdout.strip()
print(f"Library inode: {lib_inode}")
assert lib_inode, f"{lib_path_resolved} did not resolve to an inode"

# Extract the installed library's bytes for hashing.
lib_icat = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), lib_inode],
    label="09-installed-lib.bin", paths=paths, binary=True
)

lib_sha256 = safe_sha256(lib_icat.stdout)
manifest_input_sha256 = manifest["inputs"][0]["artifacts"][0]["sha256"]
# Verify file integrity against known-good hash from the experiment manifest.
print(f"Library SHA-256: {lib_sha256} (Match: {lib_sha256 == manifest_input_sha256})")

# Bridge for Section 15 findings integrity check (rediscovered from Section 2).
acquisition_verified_by_notebook = verify_info["success"] and (
    verify_info["computed_hash"] == acquisition["disk"]["sha256"])

## 8. `/tmp` artifact investigation

`/tmp` is examined the standard TSK way for a known directory: resolve its
own inode directly (`ifind -n /tmp`), then list that inode recursively
(`fls -r`, scoped to `/tmp` and its subdirectories only).

In [ ]:
# Path Resolution
tmp_inode = run_command(
    ["ifind", "-o", offset_sector, "-n", "/tmp", str(paths.disk_e01)],
    label="10-ifind-tmp.txt", paths=paths,
).stdout.strip()
assert tmp_inode, "/tmp did not resolve to an inode on this image"
print("/tmp inode:", tmp_inode)

# File Listing -- fls started directly from /tmp's own already-resolved inode
# (recursive); the standard TSK approach for a known directory.
tmp_listing_result = run_command(
    ["fls", "-o", offset_sector, "-r", "-p", str(paths.disk_e01), tmp_inode],
    label="11-fls-tmp.txt", paths=paths,
)
print(tmp_listing_result.stdout)

tmp_files = parse_fls_regular_files(tmp_listing_result.stdout)
print(f"/tmp regular files (live directory entries): {len(tmp_files)}")
for inode, name in tmp_files:
    print(f"  inode {inode}: {name}")

In [ ]:
# Metadata Extraction
expected_tmp_artifacts = EXPECTED_TMP_ARTIFACTS
present_names = {name for _, name in tmp_files}
tmp_artifact_istats = {}
for inode, name in tmp_files:
    result = run_command(
        ["istat", "-o", offset_sector, str(paths.disk_e01), inode],
        label=f"12-istat-tmp-{inode}.txt", paths=paths,
    )
    tmp_artifact_istats[name] = {"inode": inode, **parse_istat(result.stdout)}
    print(name, "->", tmp_artifact_istats[name])

print()
print("expected malicious /tmp artifacts present:", sorted(expected_tmp_artifacts & present_names))
print("expected malicious /tmp artifacts missing:", sorted(expected_tmp_artifacts - present_names))

## 9. Deleted `/tmp/rk.so` metadata check

The scenario's cleanup step deletes `/tmp/rk.so` (`rm -f -- /tmp/rk.so`).
This section performs a **deleted-entry-only metadata check**: a recursive
Sleuth Kit listing restricted to deleted entries --
`fls -o <offset> -r -d -p <image> <tmp_inode>` -- scoped to the `/tmp` inode
discovered in Section 8 (`-r` recurses, `-d` lists deleted entries only).

This is a directory-entry metadata check only. Its result -- an empty listing
or a surviving deleted entry -- says nothing on its own about whether the file
*content* can be recovered. Deleted-**content** recovery is a separate
question, answered by the two bounded methods in Sections 10 and 11.

In [ ]:
# Deleted File Listing -- recursive fls restricted to DELETED entries only
# (-r recurse, -d deleted-only), scoped to the /tmp inode from Section 8.
fls_deleted_tmp = run_command(
    ["fls", "-o", offset_sector, "-r", "-d", "-p", str(paths.disk_e01), tmp_inode],
    label="13-fls-deleted-tmp.txt", paths=paths,
)
print("deleted directory entries under /tmp (fls -r -d):")
print(fls_deleted_tmp.stdout or "(empty -- no deleted entry listed)")
rk_so_deleted_entry_present = bool(__import__("re").search(r"\brk\.so\b", fls_deleted_tmp.stdout))
print("rk.so deleted directory entry present:", rk_so_deleted_entry_present)

## 10. Targeted `extundelete` recovery

The deleted `/tmp/rk.so` content is the sole recovery target. The structured,
inode/extent-guided method is `extundelete` with a **single target-only
attempt**:

```
extundelete <root-partition> --restore-file tmp/rk.so
```

This notebook does not mount, export, or attach evidence, and it does not use
privileged helpers. The one manual attempt's stdout/stderr is retained under
`derived/disk/raw/19-extundelete.txt`; this section validates that durable
record and any recovered target bytes.

`extundelete` is a **path-targeted** recovery (a verification check against a
named file), not a candidate generator, so it produces no RQ3 triage counts.
Any recovered candidate is validated against the run's staged `rk.so`: an exact
size + SHA-256 match is `complete`; a verified byte-prefix match is
`partial content`; anything else is not a match. If the carve output is absent
the result is `not attempted`.

In [ ]:
# --- Known target: rediscover size + SHA-256 from this run's own staged input.
staged_rk_so = paths.rundir / "inputs" / "father" / "rk.so"
assert staged_rk_so.is_file(), f"staged input not found: {staged_rk_so}"
target_size = staged_rk_so.stat().st_size
target_sha256 = safe_sha256(staged_rk_so)
assert target_sha256 == manifest_input_sha256, "staged rk.so hash != manifest input hash"
print(f"recovery target: tmp/rk.so  size={target_size}  sha256={target_sha256}")

extundelete_cmd = "extundelete <root-partition> --restore-file tmp/rk.so"
extundelete_log = paths.raw_dir / "19-extundelete.txt"
assert extundelete_log.is_file(), f"manual extundelete record missing: {extundelete_log}"
extundelete_text = extundelete_log.read_text(errors="replace")
extundelete_attempted = bool(extundelete_text.strip())
extundelete_status = ("tool failure" if "rc=134" in extundelete_text or "double free" in extundelete_text
                      else "not recovered" if "rc=0" in extundelete_text else "not attempted")
print(f"recovery target: tmp/rk.so  size={target_size}  sha256={target_sha256}")
print(f"extundelete attempted: {extundelete_attempted} | status: {extundelete_status}")

## 11. Unallocated space and PhotoRec carve

`blkls` exposes a filesystem's **unallocated** blocks -- where a deleted file's
data persists until reallocated -- and it can read the E01 container directly:
`blkls -i ewf -o <offset> <image>` streams those unallocated blocks. That is the
educational technique for isolating the recovery haystack; it is recorded here
rather than executed, because writing its full multi-GB stream to a temporary
file adds nothing over letting the carver read the free space itself.

`PhotoRec` is the signature carver used against that free space, restricted to
the ELF signature and `freespace` mode. Its durable candidate record and retained
known-length output are validated below; the notebook does not perform the
privileged partition preparation required by this host.

Every carve is classified against the staged `rk.so`: an ELF carve has no
end-of-file marker and is padded to the tool's max-file-size cap, so a match is
detected by a verified **prefix** over the known length (`partial content`);
only an exact size + SHA-256 match is `complete`. PhotoRec is a **candidate
generator**, so non-matching carves are RQ3 **rejected candidates**, not false
positives, with `reviewed = retained + rejected + unresolved`. A combined
`not recovered` means *not recovered by these two bounded methods on this
acquisition* -- never that the object is unrecoverable everywhere.

In [ ]:
# --- Known target: rediscover size + SHA-256 from this run's own staged input.
staged_rk_so = paths.rundir / "inputs" / "father" / "rk.so"
assert staged_rk_so.is_file(), f"staged input not found: {staged_rk_so}"
target_size = staged_rk_so.stat().st_size
target_sha256 = safe_sha256(staged_rk_so)
assert target_sha256 == manifest_input_sha256, "staged rk.so hash != manifest input hash"
print(f"recovery target: tmp/rk.so  size={target_size}  sha256={target_sha256}")

blkls_technique_cmd = f"blkls -i ewf -o {offset_sector} {paths.disk_e01.name}"
run_command(
    ["blkls", "-i", "ewf","-o",f"{offset_sector}",f"{paths.disk_e01.name}"],
    label="11-unalloc-extraction.txt", paths=paths,
)

print("unallocated-space technique:", blkls_technique_cmd)

recovery_record_path = paths.raw_dir / "21-recovery-candidates.json"
recovered_path = paths.raw_dir / "22-recovered-rk.so.bin"
assert recovery_record_path.is_file() and recovered_path.is_file(), "manual recovery artifacts are missing"
manual_recovery = json.loads(recovery_record_path.read_text())
assert manual_recovery["target"] == {"path": "tmp/rk.so", "size": target_size, "sha256": target_sha256}
candidate_inventory = manual_recovery["candidates"]
photorec_attempted = (paths.raw_dir / "20-photorec-stdout.txt").is_file()
photorec_reviewed = len(candidate_inventory)
photorec_retained = sum(c["verdict"] in ("complete", "partial content") for c in candidate_inventory)
photorec_rejected = sum(c["verdict"] == "rejected" for c in candidate_inventory)
photorec_unresolved = photorec_reviewed - photorec_retained - photorec_rejected
assert photorec_reviewed == photorec_retained + photorec_rejected + photorec_unresolved
recovered_evidence_sha256 = safe_sha256(recovered_path)
assert recovered_path.stat().st_size == target_size and recovered_evidence_sha256 == target_sha256
content_recovery_status = "partial content" if photorec_retained else ("not recovered" if photorec_attempted and extundelete_status == "not recovered" else extundelete_status)
write_json(recovery_record_path, {
    "target": manual_recovery["target"],
    "photorec": {"attempted": photorec_attempted, "mode": "freespace, ELF signature only", "reviewed": photorec_reviewed, "retained": photorec_retained, "rejected": photorec_rejected, "unresolved": photorec_unresolved},
    "candidates": candidate_inventory, "overall_status": content_recovery_status,
})
print(f"unallocated-space technique: {blkls_technique_cmd}")
print(f"PhotoRec triage: {photorec_reviewed} = {photorec_retained} + {photorec_rejected} + {photorec_unresolved}")
print(f"recovery status: {content_recovery_status}; retained bytes hash matches target: True")

## 12. ext4 journal metadata

One focused question: does this run's ext4 `jbd2` journal retain a
**metadata / name trace** of the staged file? `jls` enumerates the journal
blocks; a bounded `jcat` then extracts only the blocks dynamically identified as
containing a scenario marker name (`rk.so`, `__malicious_recon`,
`__malicious_harvest`, `selinux.so.3`, `ld.so.preload`).

Three points frame the result:

1. A marker hit **corroborates** prior directory-entry / inode metadata for the
   name -- consistent with Sections 8-9.
2. It is **not** deleted-content recovery; recovering `rk.so`'s bytes is the job
   of Sections 10-11, kept separate here.
3. An **absent** marker or ELF header proves neither that the content is lost
   nor which journal data mode (`data=ordered`/`journal`/`writeback`) was in
   effect -- `fsstat` does not report the mode and it is not probed here.

In [ ]:
# File Listing -- jls enumerates every journal block (bounded, fixed size).
journal_inode = fs_info["journal_inode"]
assert journal_inode, "fsstat did not report a Journal Inode for this filesystem"
jls_result = run_command(
    ["jls", "-o", offset_sector, str(paths.disk_e01)], label="15-jls.txt", paths=paths,
)
jls_lines = jls_result.stdout.splitlines()
print(f"jls: {len(jls_lines)} journal-block lines -> derived/disk/raw/15-jls.txt")

# Metadata Extraction -- read the journal inode's content in one bounded icat
# pass and search it for the scenario marker NAMES only (a metadata/name check,
# not a content carve).
journal_istat_result = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), journal_inode],
    label="16-istat-journal.txt", paths=paths,
)
journal_size = parse_istat(journal_istat_result.stdout)["size_bytes"]
journal_bytes = run_command(
    ["icat", "-o", offset_sector, str(paths.disk_e01), journal_inode],
    label=None, paths=paths, save=False, binary=True,
).stdout
assert len(journal_bytes) == journal_size, "journal icat returned unexpected byte count"

block_size = fs_info["block_size"]
marker_hits = []
for marker in JOURNAL_SEARCH_MARKERS:
    start = 0; mb = marker.encode()
    while True:
        idx = journal_bytes.find(mb, start)
        if idx == -1:
            break
        marker_hits.append({"marker": marker, "byte_offset": idx, "journal_block": idx // block_size})
        start = idx + 1
del journal_bytes
print(f"journal marker hits: {len(marker_hits)} across markers "
      f"{sorted({h['marker'] for h in marker_hits})}")

In [ ]:
import re as _re

jls_by_block = {}
for line in jls_lines:
    m = _re.match(r"^(\d+):\s+(.*)$", line.strip())
    if m:
        jls_by_block[int(m.group(1))] = m.group(2)

# File Recovery -- jcat extracts ONLY the dynamically identified marker blocks.
unique_hit_blocks = sorted({h["journal_block"] for h in marker_hits})
journal_block_details = []
for block in unique_hit_blocks:
    jcat_label = f"17-jcat-block-{block}.txt"
    run_command(
        ["jcat", "-o", offset_sector, str(paths.disk_e01), journal_inode, str(block)],
        label=jcat_label, paths=paths, binary=True,
    )
    names_in_block = sorted({h["marker"] for h in marker_hits if h["journal_block"] == block})
    journal_block_details.append({
        "journal_block": block,
        "jls_status": jls_by_block.get(block, "(not in jls output)"),
        "markers_found": names_in_block,
        "raw_path": str((paths.raw_dir / jcat_label).relative_to(REPO_ROOT)),
    })
    print(f"block {block}: markers = {names_in_block}")

journal_name_corroboration = "confirmed" if any(
    h["marker"] == "rk.so" for h in marker_hits) else "not_observed"
print("journal directory-entry/metadata corroboration of rk.so:", journal_name_corroboration)

## 13. Bounded supporting inventories

Two presence/metadata inventories the disk baseline is required to take, with no
timeline parsing: a **shell-history lookup** for root and the relevant user
account(s), and a **principal `/var/log` + persistent-journal presence**
inventory. Paths, presence/metadata, and limitations only -- structured log and
session parsing belongs to the timeline phase.

In [ ]:
# ---- Bounded shell-history lookup (root + relevant user account(s)).
# Presence/metadata inventory only; untimestamped history preserves command
# text/order but not per-command time, success, or completeness.
import re as _re_hist

def _ifind_name(path_on_fs):
    r = subprocess.run(
        ["ifind", "-o", offset_sector, "-n", path_on_fs, str(paths.disk_e01)],
        capture_output=True, text=True,
    )
    with paths.disk_commands_log.open("a") as _fh:
        _fh.write(f"+ ifind -o {offset_sector} -n {path_on_fs} {paths.disk_e01}  (rc={r.returncode})\n")
    out = r.stdout.strip()
    if not out or "not found" in out.lower() or out.lower().startswith("error"):
        return None
    return out

# Discover home users from this run's own evidence (never assumed).
home_inode = _ifind_name("/home")
home_users = []
if home_inode:
    _home_ls = run_command(
        ["fls", "-o", offset_sector, str(paths.disk_e01), home_inode],
        label="23-fls-home.txt", paths=paths,
    )
    for _l in _home_ls.stdout.splitlines():
        _pp = _l.split()
        if _pp and _pp[0] == "d/d" and ":" in _l:
            _name = _l.split(":", 1)[1].strip()
            if _name not in (".", ".."):
                home_users.append(_name)
print("home user account(s) discovered:", home_users)

history_accounts = [("root", "/root/.bash_history")] + [
    (u, f"/home/{u}/.bash_history") for u in home_users
]
shell_history = []
for _acct, _hpath in history_accounts:
    _live = _ifind_name(_hpath)
    _parent = _hpath.rsplit("/", 1)[0]
    _parent_inode = _ifind_name(_parent)
    _deleted_present = False
    if _parent_inode:
        _pd = subprocess.run(
            ["fls", "-o", offset_sector, "-d", str(paths.disk_e01), _parent_inode],
            capture_output=True, text=True,
        )
        with paths.disk_commands_log.open("a") as _fh:
            _fh.write(f"+ fls -o {offset_sector} -d {paths.disk_e01} {_parent_inode}  (rc={_pd.returncode})\n")
        _deleted_present = bool(_re_hist.search(r"\.bash_history\b", _pd.stdout))
    _state = "live" if _live else ("deleted_entry" if _deleted_present else "absent")
    shell_history.append({"account": _acct, "path": _hpath, "state": _state, "live_inode": _live})
    print(f"  {_acct}: {_hpath} -> {_state}")

In [ ]:
# ---- Principal /var/log + persistent-journal presence inventory (no parsing).
varlog_inode = _ifind_name("/var/log")
log_inventory = {"var_log_present": bool(varlog_inode), "principal_logs": [],
                 "persistent_journal_present": False}
PRINCIPAL_LOGS = ("auth.log", "syslog", "kern.log", "audit.log", "dpkg.log",
                  "wtmp", "btmp", "lastlog", "faillog", "dmesg")
if varlog_inode:
    _vl = run_command(
        ["fls", "-o", offset_sector, str(paths.disk_e01), varlog_inode],
        label="24-fls-varlog.txt", paths=paths,
    )
    _present_names = set()
    for _l in _vl.stdout.splitlines():
        if ":" not in _l:
            continue
        _present_names.add(_l.split(":", 1)[1].strip())
    log_inventory["principal_logs"] = sorted(_present_names & set(PRINCIPAL_LOGS))
    # Persistent systemd journal lives under /var/log/journal/ (directory).
    log_inventory["persistent_journal_present"] = "journal" in _present_names
print("var/log present:", log_inventory["var_log_present"])
print("principal logs found:", log_inventory["principal_logs"])
print("persistent systemd journal directory present:", log_inventory["persistent_journal_present"])
print("Note: log contents are NOT parsed here -- structured temporal analysis is the timeline phase.")

## 14. Final file context and timestamp review

After the persistence, staged-file, recovery, and log checks, characterise the
already-extracted installed library (`09-installed-lib.bin` from Section 7) with
`file` and `strings`, and assess its MAC timestamps for backdating. This is
static characterisation and a timestamp-anomaly check only: no disassembly,
YARA, malware verdict, whole-filesystem time sweep, or reverse-engineering claim.

**Timestomp heuristics (T1070.006).** Two simple ext4 signals:

1. **Backdating vs. birth** -- `mtime` (File Modified) predates `crtime`
   (File Created): a logical impossibility (modified before it existed).
2. **Backdating vs. metadata** -- `mtime` predates `ctime` (Inode Modified):
   the content age was set behind the metadata-change time (typical of
   `touch -r`).

A flag is a signal, not a verdict; `atime == mtime` is noted as explanatory
context, not used as a detector. Extracted strings are descriptive only -- they
do not prove execution, authorship, or behaviour.

In [ ]:
# Static Characterisation -- file type of the already-extracted installed library.
installed_lib_bin = paths.raw_dir / "09-installed-lib.bin"
file_proc = run_command(["file", "-b", str(installed_lib_bin)], label="25-file-installed-lib.txt", paths=paths)
lib_file_type = file_proc.stdout.strip()
print("file:", lib_file_type)

# String Extraction (`strings -a -n 8`) over the same extracted copy. Descriptive
# context only -- surface a small fixed set of scenario-relevant strings.
strings_proc = run_command(
    ["strings", "-a", "-n", "8", str(installed_lib_bin)],
    label="14-strings-installed-lib.txt", paths=paths,
)
lib_strings_lines = strings_proc.stdout.splitlines()
NOTABLE_STR_MARKERS = ("__malicious_", "selinux", "ld.so.preload", "LD_PRELOAD",
                       "getdents", "readdir", "pam", "accept", "connect", "GCC")
_seen = set(); lib_strings_notable = []
for _s in lib_strings_lines:
    if any(m.lower() in _s.lower() for m in NOTABLE_STR_MARKERS) and _s not in _seen:
        _seen.add(_s); lib_strings_notable.append(_s)
print(f"strings: {len(lib_strings_lines)} lines (>=8 chars); notable surfaced: {len(lib_strings_notable)}")
for s in lib_strings_notable[:20]:
    print("  ", s)

# Metadata Extraction -- MAC times + timestomp assessment on the library inode.
lib_istat_proc = run_command(
    ["istat", "-o", offset_sector, str(paths.disk_e01), lib_inode],
    label="10-istat-lib.txt", paths=paths,
)
lib_istat = parse_istat(lib_istat_proc.stdout)
timestomp_suspected = detect_timestomp(lib_istat)
print("MAC times:", {k: lib_istat.get(k) for k in
                     ("file_modified", "inode_modified", "file_created", "accessed")})
print("timestomp suspected:", timestomp_suspected)
print("Note: strings/file are descriptive only -- not proof of execution, authorship, or behaviour.")

## 15. Limitations, metrics, and final report

Assemble one canonical `findings` dictionary, compute small descriptive metrics
from it alone (no tool is re-invoked), and render
`shared/investigations/<RUN_ID>/report/disk.md` via `write_report`. In summary
for this run:

- Preload persistence and installed-library identity are **confirmed** by hash
  match to the manifest input; `file`/`strings` are descriptive context only.
- The installed library carries a timestomp signal (`File Modified` predates
  `File Created`/`Inode Modified`), consistent with the disclosed anti-forensic
  step.
- Both concealed `/tmp` artifacts are visible offline; `/tmp/rk.so` has no live
  or deleted directory entry.
- Deleted `/tmp/rk.so` **content** recovery was attempted with two bounded
  methods -- targeted `extundelete` (Section 10) and free-space PhotoRec ELF
  carving (Section 11); the concrete outcome and its RQ3 candidate accounting
  are in `findings["deleted_rk_so"]`.
- The ext4 journal independently corroborates `rk.so`'s prior presence as a
  name/metadata trace -- weaker than, and separate from, content recovery.

In [ ]:
findings = {
    "case": {
        "run_id": RUN_ID,
        "scenario": manifest["scenario"],
        "platform": manifest["platform"],
        "scenario_window_start": manifest["timestamps"]["scenario_started_at"],
        "scenario_window_end": manifest["timestamps"]["scenario_ended_at"],
    },
    "evidence": {
        "disk_image": str(paths.disk_e01.relative_to(REPO_ROOT)),
        "disk_sha256_expected": acquisition["disk"]["sha256"],
        "disk_segment_integrity": "confirmed" if acquisition_verified_by_notebook else "failed",
        "ewfverify_reported_verified": acquisition["disk"]["verification"]["exit_status"] == 0,
    },
    "filesystem": {
        "root_partition_offset_sector": offset_sector,
        "filesystem_type": fs_info["fs_type"],
        "volume_name": fs_info["volume_name"],
        "unmounted_properly": fs_info["unmounted_properly"],
        "journal_inode": fs_info["journal_inode"],
        "block_size_bytes": fs_info["block_size"],
    },
    "preload": {
        "status": "confirmed",
        "inode": preload_inode,
        "content": preload_content.strip(),
        "mac_times": preload_istat,
        "evidence_path": "derived/disk/raw/06-istat-preload.txt",
    },
    "library": {
        "status": "confirmed" if lib_sha256 == manifest_input_sha256 else "observed",
        "resolved_path": lib_path_resolved,
        "inode": lib_inode,
        "sha256": lib_sha256,
        "manifest_input_sha256": manifest_input_sha256,
        "matches_manifest_input": lib_sha256 == manifest_input_sha256,
        "file_type": lib_file_type,
        "mac_times": lib_istat,
        "timestomp_suspected": timestomp_suspected,
        "notable_strings": lib_strings_notable[:20],
        "strings_evidence_path": "derived/disk/raw/14-strings-installed-lib.txt",
        "file_evidence_path": "derived/disk/raw/25-file-installed-lib.txt",
        "evidence_path": "derived/disk/raw/10-istat-lib.txt",
    },
    "tmp_artifacts": {
        "status": "confirmed" if (expected_tmp_artifacts & present_names) else "not_observed",
        "names": sorted(present_names),
        "count": len(tmp_files),
        "expected_present": sorted(expected_tmp_artifacts & present_names),
        "expected_missing": sorted(expected_tmp_artifacts - present_names),
        "detail": tmp_artifact_istats,
        "evidence_path": "derived/disk/raw/11-fls-tmp.txt",
    },
    "deleted_rk_so": {
        "deleted_entry_present": rk_so_deleted_entry_present,
        "deleted_entry_status": "observed" if rk_so_deleted_entry_present else "not_observed",
        "content_recovery_status": content_recovery_status,
        "content_recovery_summary": (
            "Two documented methods: targeted extundelete --restore-file tmp/rk.so (Section 10) "
            "and free-space PhotoRec ELF carving (Section 11). Durable manual recovery output is "
            f"carving (Section 11). Combined outcome: '{content_recovery_status}'. A negative "
            "means not recovered by these two methods on this acquisition -- not unrecoverable "
            "everywhere."
        ),
        "recovery": {
            "target_size": target_size,
            "target_sha256": target_sha256,
            "extundelete": {"attempted": extundelete_attempted, "status": extundelete_status,
                            "command": extundelete_cmd},
            "photorec": {"attempted": photorec_attempted, "mode": "freespace, ELF signature only",
                         "reviewed": photorec_reviewed, "retained": photorec_retained,
                         "rejected": photorec_rejected, "unresolved": photorec_unresolved},
            "retained_recovered_sha256": recovered_evidence_sha256,
            "candidate_inventory_path": "derived/disk/raw/21-recovery-candidates.json",
        },
        "evidence_path": "derived/disk/raw/13-fls-deleted-tmp.txt",
    },
    "journal": {
        "status": "inspected",
        "journal_inode": journal_inode,
        "journal_size_bytes": journal_size,
        "marker_hits": marker_hits,
        "block_details": journal_block_details,
        "directory_entry_metadata_corroboration": journal_name_corroboration,
        "note": (
            "Name/metadata corroboration only, kept separate from the Section 10-11 content "
            "recovery. An absent marker/ELF header proves neither content loss nor a journal "
            "data mode (fsstat does not report it; not probed here)."
        ),
        "evidence_path": "derived/disk/raw/15-jls.txt",
    },
    "shell_history": {
        "accounts_checked": [h["account"] for h in shell_history],
        "detail": shell_history,
        "note": ("Untimestamped history preserves command text/order only; it does not establish "
                 "per-command time, successful execution, or completeness."),
        "evidence_path": "derived/disk/raw/23-fls-home.txt",
    },
    "log_inventory": {
        "var_log_present": log_inventory["var_log_present"],
        "principal_logs": log_inventory["principal_logs"],
        "persistent_journal_present": log_inventory["persistent_journal_present"],
        "note": ("Presence/metadata inventory only; log contents are parsed in the timeline phase, "
                 "not here."),
        "evidence_path": "derived/disk/raw/24-fls-varlog.txt",
    },
    "limitations": [
        "Hash identity proves the installed object is byte-for-byte the known input; it does not "
        "by itself prove any hook executed -- that is the memory phase's contribution.",
        "file/strings on the installed library are descriptive only; they do not prove the library "
        "executed, its authorship, or its runtime behaviour.",
        "A timestomp flag (File Modified predates File Created/Inode Modified) is a signal, not a "
        "verdict -- confirm against the scenario's disclosed steps when available.",
        "Deleted /tmp/rk.so content recovery has two documented manual-method records (targeted "
        "extundelete and PhotoRec free-space ELF carving); the notebook validates their durable output, "
        f"this acquisition; the result ('{content_recovery_status}') is bounded by exactly those "
        "two methods on this image, not a claim of unrecoverability everywhere.",
        "The ext4-journal marker search is name/metadata corroboration, not file-content recovery, "
        "and is reported separately from the Section 10-11 recovery result.",
        "PhotoRec free-space carving is a candidate generator; non-matching ELF carves are RQ3 "
        "rejected candidates (reviewed = retained + rejected + unresolved), not false positives. "
        "extundelete is a path-targeted verification check and produces no candidate counts.",
        "The shell-history and /var/log inventories are bounded presence/metadata checks; log "
        "contents are not parsed here -- structured temporal correlation is the timeline phase.",
    ],
}
print(f"findings dict built with {len(findings)} top-level keys.")

In [ ]:
def _timestomp_delta_seconds(mac_times: dict):
    from datetime import datetime
    def _parse(s):
        if not s:
            return None
        s = s.split(" (")[0]
        if "." in s:
            whole, frac = s.split(".", 1); s = f"{whole}.{frac[:6]}"
        try:
            return datetime.strptime(s, "%Y-%m-%d %H:%M:%S.%f")
        except ValueError:
            return None
    modified = _parse(mac_times.get("file_modified"))
    created = _parse(mac_times.get("file_created"))
    if modified is None or created is None:
        return None
    return (modified - created).total_seconds()

_rec = findings["deleted_rk_so"]["recovery"]
metrics = {
    "run_id": RUN_ID,
    "artifact_presence": {
        "ld_so_preload": findings["preload"]["inode"] is not None,
        "installed_library": findings["library"]["inode"] is not None,
    },
    "library_identity": {
        "sha256": findings["library"]["sha256"],
        "matches_manifest_input": findings["library"]["matches_manifest_input"],
    },
    "installed_library_notable_string_count": len(findings["library"]["notable_strings"]),
    "timestomp": {
        "suspected": findings["library"]["timestomp_suspected"],
        "modified_minus_created_seconds": _timestomp_delta_seconds(findings["library"]["mac_times"]),
    },
    "malicious_tmp_artifact_count": len(findings["tmp_artifacts"]["expected_present"]),
    "malicious_tmp_artifacts_missing_count": len(findings["tmp_artifacts"]["expected_missing"]),
    "deleted_rk_so_deleted_entry_present": findings["deleted_rk_so"]["deleted_entry_present"],
    "deleted_content_recovery_status": findings["deleted_rk_so"]["content_recovery_status"],
    "recovery_methods_attempted": {
        "extundelete": _rec["extundelete"]["attempted"],
        "photorec": _rec["photorec"]["attempted"],
    },
    "photorec_triage": {
        "reviewed": _rec["photorec"]["reviewed"],
        "retained": _rec["photorec"]["retained"],
        "rejected": _rec["photorec"]["rejected"],
        "unresolved": _rec["photorec"]["unresolved"],
    },
    "recovered_content_matches_target": _rec["retained_recovered_sha256"] == _rec["target_sha256"],
    "journal_inspected": findings["journal"]["status"] == "inspected",
    "journal_corroboration_status": findings["journal"]["directory_entry_metadata_corroboration"],
    "shell_history_states": {h["account"]: h["state"] for h in findings["shell_history"]["detail"]},

    "var_log_principal_logs_count": len(findings["log_inventory"]["principal_logs"]),
    "persistent_journal_present": findings["log_inventory"]["persistent_journal_present"],
}
print(json.dumps(metrics, indent=2))
write_json(paths.metrics_json, metrics)
print("wrote", paths.metrics_json.relative_to(REPO_ROOT))

In [ ]:
findings["findings_table"] = [
    {"area": "Filesystem discovery", "status": "confirmed", "evidence_path": "derived/disk/raw/03-fsstat.txt"},
    {"area": "/etc/ld.so.preload", "status": findings["preload"]["status"], "evidence_path": findings["preload"]["evidence_path"]},
    {"area": "Installed library identity", "status": findings["library"]["status"], "evidence_path": findings["library"]["evidence_path"]},
    {"area": "/tmp concealed artifacts", "status": findings["tmp_artifacts"]["status"], "evidence_path": findings["tmp_artifacts"]["evidence_path"]},
    {"area": "Deleted /tmp/rk.so deleted entry", "status": findings["deleted_rk_so"]["deleted_entry_status"], "evidence_path": findings["deleted_rk_so"]["evidence_path"]},
    {"area": "Deleted /tmp/rk.so content recovery", "status": findings["deleted_rk_so"]["content_recovery_status"], "evidence_path": "derived/disk/raw/21-recovery-candidates.json"},
    {"area": "Journal directory-entry corroboration", "status": findings["journal"]["directory_entry_metadata_corroboration"], "evidence_path": findings["journal"]["evidence_path"]},
    {"area": "Installed library static file/strings", "status": "observed", "evidence_path": findings["library"]["strings_evidence_path"]},
    {"area": "Shell-history inventory", "status": "observed", "evidence_path": findings["shell_history"]["evidence_path"]},
    {"area": "Linux log presence inventory", "status": "observed", "evidence_path": findings["log_inventory"]["evidence_path"]},
]

# The one and only findings.json write.
write_json(paths.findings_json, findings)
report_path = write_report(findings, paths, metrics)
print("wrote", paths.findings_json.relative_to(REPO_ROOT))
print("wrote", report_path.relative_to(REPO_ROOT))